In [4]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls '/content/drive/MyDrive/KQ-Capstone/dataset/raw/'


Alerts.gsheet		 Mobilization_Log.gsheet  Vehicles.gsheet
Crew_Profiles.gsheet	 Transit_Events.gsheet
Flight_Schedules.gsheet  Users.gsheet


In [6]:
!find /content/drive/MyDrive -iname "*ransit*"

/content/drive/MyDrive/KQ-Capstone/dataset/raw/Transit_Events.gsheet


In [7]:
from google.colab import auth
auth.authenticate_user()

import gspread
import pandas as pd
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

In [8]:
SHEET_NAMES = {
    "users": "Users",
    "crew_profiles": "Crew_Profiles",
    "vehicles": "Vehicles",
    "flight_schedules": "Flight_Schedules",
    "transit_events": "Transit_Events",
    "mobilization_log": "Mobilization_Log",
    "alerts": "Alerts",
}

def load_sheet(name: str) -> pd.DataFrame:
    sh = gc.open(name)          # finds it by title, no ID needed
    ws = sh.sheet1
    return pd.DataFrame(ws.get_all_records())

dfs = {key: load_sheet(name) for key, name in SHEET_NAMES.items()}

for key, df in dfs.items():
    print(key, df.shape)

dfs["transit_events"].head()

users (195, 5)
crew_profiles (180, 5)
vehicles (40, 4)
flight_schedules (2000, 7)
transit_events (8000, 22)
mobilization_log (32000, 4)
alerts (2702, 6)


,transit_id,crew_id,flight_id,vehicle_id,corridor,pickup_location,dropoff_location,scheduled_pickup_time,actual_pickup_time,scheduled_arrival_time,...,day_of_week,hour_of_day,is_weekday,is_rush_hour,weather_flag,traffic_congestion_index,historical_avg_delay_min,delay_minutes,delay_risk_class,split
0,4344,29,49,9,KQ Training Centre - JKIA,KQ Training Centre,JKIA,2025-01-10 17:27:19,2025-01-10 17:29:51,2025-01-10 17:39:30,...,4,17,1,1,1,0.590,18.37,54.76,CRITICAL,train
1,2681,51,1936,36,Westlands - JKIA,Westlands,JKIA,2025-02-04 2:43:38,2025-02-04 2:42:35,2025-02-04 3:48:23,...,1,2,1,0,0,0.499,24.32,46.65,CRITICAL,train
2,3762,123,241,19,South B - JKIA,South B,JKIA,2025-01-23 10:20:10,2025-01-23 10:24:07,2025-01-23 10:42:41,...,3,10,1,0,0,0.477,8.00,81.28,CRITICAL,train
3,4998,30,1576,18,Medical Centre - JKIA,Medical Centre,JKIA,2025-02-15 12:00:21,2025-02-15 12:00:51,2025-02-15 12:23:11,...,5,12,0,0,0,0.332,13.46,96.65,CRITICAL,train
4,6132,82,481,19,Westlands - JKIA,Westlands,JKIA,2025-03-23 18:43:40,2025-03-23 18:45:54,2025-03-23 19:55:34,...,6,18,0,1,0,0.634,29.09,48.95,CRITICAL,train


In [9]:
for key, df in dfs.items():
    print(f"--- {key} ---")
    print(df.columns.tolist())
    print(df.dtypes)
    print()

--- users ---
['user_id', 'full_name', 'role', 'email', 'phone']
user_id       int64
full_name    object
role         object
email        object
phone         int64
dtype: object

--- crew_profiles ---
['crew_id', 'user_id', 'rank', 'base_location', 'employment_start_date']
crew_id                   int64
user_id                   int64
rank                     object
base_location            object
employment_start_date    object
dtype: object

--- vehicles ---
['vehicle_id', 'vehicle_type', 'capacity', 'status']
vehicle_id       int64
vehicle_type    object
capacity         int64
status          object
dtype: object

--- flight_schedules ---
['flight_id', 'crew_id', 'flight_number', 'scheduled_departure', 'scheduled_arrival', 'origin', 'destination']
flight_id               int64
crew_id                 int64
flight_number          object
scheduled_departure    object
scheduled_arrival      object
origin                 object
destination            object
dtype: object

--- transit_

In [10]:
transit = dfs["transit_events"].copy()

# Fix datetime columns
for col in ["scheduled_pickup_time", "actual_pickup_time",
            "scheduled_arrival_time", "actual_arrival_time"]:
    transit[col] = pd.to_datetime(transit[col])

crew = dfs["crew_profiles"].copy()
vehicles = dfs["vehicles"].copy()
flights = dfs["flight_schedules"].copy()
alerts = dfs["alerts"].copy()
alerts["resolved"] = alerts["resolved"].astype(str).str.upper() == "TRUE"

print(transit.dtypes)

transit_id                           int64
crew_id                              int64
flight_id                            int64
vehicle_id                           int64
corridor                            object
pickup_location                     object
dropoff_location                    object
scheduled_pickup_time       datetime64[ns]
actual_pickup_time          datetime64[ns]
scheduled_arrival_time      datetime64[ns]
actual_arrival_time         datetime64[ns]
distance_km                        float64
day_of_week                          int64
hour_of_day                          int64
is_weekday                           int64
is_rush_hour                         int64
weather_flag                         int64
traffic_congestion_index           float64
historical_avg_delay_min           float64
delay_minutes                      float64
delay_risk_class                    object
split                               object
dtype: object


In [11]:
df = (transit
      .merge(crew[["crew_id", "rank", "base_location"]], on="crew_id", how="left")
      .merge(vehicles[["vehicle_id", "vehicle_type", "capacity"]], on="vehicle_id", how="left")
      .merge(flights[["flight_id", "destination"]], on="flight_id", how="left"))

print(df.shape)
df.head()

(8000, 27)


,transit_id,crew_id,flight_id,vehicle_id,corridor,pickup_location,dropoff_location,scheduled_pickup_time,actual_pickup_time,scheduled_arrival_time,...,traffic_congestion_index,historical_avg_delay_min,delay_minutes,delay_risk_class,split,rank,base_location,vehicle_type,capacity,destination
0,4344,29,49,9,KQ Training Centre - JKIA,KQ Training Centre,JKIA,2025-01-10 17:27:19,2025-01-10 17:29:51,2025-01-10 17:39:30,...,0.590,18.37,54.76,CRITICAL,train,First Officer,JKIA Base,Shuttle Bus,20,CDG
1,2681,51,1936,36,Westlands - JKIA,Westlands,JKIA,2025-02-04 02:43:38,2025-02-04 02:42:35,2025-02-04 03:48:23,...,0.499,24.32,46.65,CRITICAL,train,Senior Cabin Crew,Embakasi Base,Crew Van,8,LOS
2,3762,123,241,19,South B - JKIA,South B,JKIA,2025-01-23 10:20:10,2025-01-23 10:24:07,2025-01-23 10:42:41,...,0.477,8.00,81.28,CRITICAL,train,Cabin Crew,Embakasi Base,Minibus,14,LHR
3,4998,30,1576,18,Medical Centre - JKIA,Medical Centre,JKIA,2025-02-15 12:00:21,2025-02-15 12:00:51,2025-02-15 12:23:11,...,0.332,13.46,96.65,CRITICAL,train,Senior Cabin Crew,Westlands Base,Shuttle Bus,20,AMS
4,6132,82,481,19,Westlands - JKIA,Westlands,JKIA,2025-03-23 18:43:40,2025-03-23 18:45:54,2025-03-23 19:55:34,...,0.634,29.09,48.95,CRITICAL,train,Senior Cabin Crew,Westlands Base,Minibus,14,JNB


In [12]:
train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

print(len(train_df), len(val_df), len(test_df))

6398 799 803


In [13]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ["corridor", "rank", "base_location", "vehicle_type", "destination"]
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le

df[[c + "_enc" for c in cat_cols]].head()

,corridor_enc,rank_enc,base_location_enc,vehicle_type_enc,destination_enc
0,1,2,1,3,3
1,5,3,0,0,9
2,4,0,0,1,8
3,2,3,2,3,1
4,5,3,2,1,7


In [14]:
feature_cols = [
    "is_weekday", "is_rush_hour", "weather_flag",
    "traffic_congestion_index", "historical_avg_delay_min",
    "distance_km", "hour_of_day", "day_of_week", "capacity",
    "corridor_enc", "rank_enc", "base_location_enc",
    "vehicle_type_enc", "destination_enc",
]

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]

X_train, y_train = train[feature_cols], train["delay_risk_class"]
X_test,  y_test  = test[feature_cols],  test["delay_risk_class"]

print(X_train.shape, X_test.shape)

(6398, 14) (803, 14)


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
print("Macro F1:", f1_score(y_test, pred, average="macro"))
print(classification_report(y_test, pred))

Macro F1: 0.3327120125607629
              precision    recall  f1-score   support

    CRITICAL       0.17      0.01      0.02        87
        HIGH       0.43      0.54      0.48       232
         LOW       0.56      0.79      0.65       298
      MEDIUM       0.29      0.13      0.18       186

    accuracy                           0.48       803
   macro avg       0.36      0.37      0.33       803
weighted avg       0.42      0.48      0.42       803



In [16]:
import pandas as pd

importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

traffic_congestion_index    0.195913
historical_avg_delay_min    0.176203
hour_of_day                 0.116579
destination_enc             0.107509
day_of_week                 0.078383
rank_enc                    0.056954
base_location_enc           0.052525
capacity                    0.041089
distance_km                 0.041000
vehicle_type_enc            0.040178
corridor_enc                0.040016
weather_flag                0.021851
is_rush_hour                0.021338
is_weekday                  0.010464
dtype: float64


In [17]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost needs the target as integers, not strings
target_le = LabelEncoder()
y_train_enc = target_le.fit_transform(y_train)
y_test_enc = target_le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric="mlogloss"
)
xgb.fit(X_train, y_train_enc)

pred_xgb = xgb.predict(X_test)
print("Macro F1 (XGBoost):", f1_score(y_test_enc, pred_xgb, average="macro"))
print(classification_report(y_test_enc, pred_xgb, target_names=target_le.classes_))

Macro F1 (XGBoost): 0.3449918499433909
              precision    recall  f1-score   support

    CRITICAL       0.14      0.02      0.04        87
        HIGH       0.45      0.56      0.50       232
         LOW       0.57      0.79      0.66       298
      MEDIUM       0.27      0.13      0.17       186

    accuracy                           0.49       803
   macro avg       0.36      0.38      0.34       803
weighted avg       0.42      0.49      0.44       803



In [18]:
df["delay_binary"] = df["delay_risk_class"].isin(["HIGH", "CRITICAL"]).astype(int)

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]
X_train, y_train_b = train[feature_cols], train["delay_binary"]
X_test,  y_test_b  = test[feature_cols],  test["delay_binary"]

xgb_bin = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, random_state=42, eval_metric="logloss")
xgb_bin.fit(X_train, y_train_b)
pred_bin = xgb_bin.predict(X_test)
print("F1 (binary):", f1_score(y_test_b, pred_bin))
print(classification_report(y_test_b, pred_bin))

F1 (binary): 0.5263157894736842
              precision    recall  f1-score   support

           0       0.69      0.76      0.73       484
           1       0.57      0.49      0.53       319

    accuracy                           0.65       803
   macro avg       0.63      0.62      0.63       803
weighted avg       0.65      0.65      0.65       803



In [19]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report

param_grid = {
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "n_estimators": [200, 300, 500],
    "min_child_weight": [1, 3, 5],
}

xgb_base = XGBClassifier(random_state=42, eval_metric="logloss")

grid = GridSearchCV(
    xgb_base, param_grid, scoring="f1", cv=3, n_jobs=-1, verbose=1
)
grid.fit(X_train, y_train_b)

print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_xgb = grid.best_estimator_
pred_best = best_xgb.predict(X_test)
print("Test F1:", f1_score(y_test_b, pred_best))
print(classification_report(y_test_b, pred_best))

Fitting 3 folds for each of 144 candidates, totalling 432 fits
Best params: {'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 500}
Best CV F1: 0.5382648963241526
Test F1: 0.576271186440678
              precision    recall  f1-score   support

           0       0.72      0.79      0.75       484
           1       0.63      0.53      0.58       319

    accuracy                           0.69       803
   macro avg       0.67      0.66      0.67       803
weighted avg       0.68      0.69      0.68       803



In [20]:
import joblib
import os
os.makedirs('/content/drive/MyDrive/KQ-Capstone/models', exist_ok=True)
joblib.dump(best_xgb, '/content/drive/MyDrive/KQ-Capstone/models/xgb_tuned_v1.pkl')

['/content/drive/MyDrive/KQ-Capstone/models/xgb_tuned_v1.pkl']

In [21]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train_b)

xgb_final = XGBClassifier(**grid.best_params_, random_state=42, eval_metric="logloss")
xgb_final.fit(X_train_res, y_train_res)
pred_final = xgb_final.predict(X_test)
print("F1 (tuned + SMOTE):", f1_score(y_test_b, pred_final))
print(classification_report(y_test_b, pred_final))

F1 (tuned + SMOTE): 0.6169296987087518
              precision    recall  f1-score   support

           0       0.76      0.66      0.71       484
           1       0.57      0.67      0.62       319

    accuracy                           0.67       803
   macro avg       0.66      0.67      0.66       803
weighted avg       0.68      0.67      0.67       803



In [ ]:
joblib.dump(xgb_final, '/content/drive/MyDrive/KQ-Capstone/models/xgb_final_v1.pkl')

In [22]:
joblib.dump(xgb_final, '/content/drive/MyDrive/KQ-Capstone/models/xgb_final_v1.pkl')

['/content/drive/MyDrive/KQ-Capstone/models/xgb_final_v1.pkl']